# Chapter 2 Lab — LLM Primitives

**Companion notebook for [Chapter 2: LLM Primitives](../part-01-foundations/ch02-llm-primitives.html)**

In this lab you will get hands-on with the atomic building blocks every agentic system relies on:

| Section | What you will do |
|---|---|
| Tokenization Explorer | Tokenize text, count tokens, compare encodings across models |
| Completion API Basics | Make raw completion calls, inspect response objects, extract usage metadata |
| Message Roles | Experiment with system / user / assistant message patterns |
| Temperature Experiments | Run the same prompt at different temperatures and compare outputs |
| Streaming | Consume a streaming response token by token |
| Tool / Function Calling | Define a tool, trigger a tool call, feed the result back |
| Structured Outputs | Get guaranteed JSON output via a Pydantic model |

> **Prerequisites:** An OpenAI API key stored in the `OPENAI_API_KEY` environment variable (or in a `.env` file in this directory).

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────
# Run once to install dependencies (restart kernel after first run)
%pip install -q openai tiktoken python-dotenv pydantic

In [ ]:
import json, os, textwrap
from dotenv import load_dotenv

import tiktoken
from openai import OpenAI

load_dotenv()  # reads .env if present

client = OpenAI()  # uses OPENAI_API_KEY from environment
MODEL = "gpt-4o"   # default model for all calls in this notebook

print(f"Using model: {MODEL}")
print(f"API key loaded: {'yes' if os.getenv('OPENAI_API_KEY') else 'NO — set OPENAI_API_KEY'}")

---
## 1. Tokenization Explorer

Tokenizers are **separate artifacts** from the model. You can (and should) count tokens locally before making any API call. Different models use different tokenizers, so the same text produces different token counts.

Key insight from Chapter 2: English prose averages ~1.3 tokens/word, code ~2.5 tokens/word, and JSON is notoriously token-hungry.

In [ ]:
# ── 1a. Basic tokenization ────────────────────────────────────────────────
encoding = tiktoken.encoding_for_model("gpt-4o")

text = "Explain the mechanism of action for metformin in Type 2 diabetes."
tokens = encoding.encode(text)

print(f"Text:        {text}")
print(f"Token count: {len(tokens)}")
print(f"Token IDs:   {tokens}")
print(f"Decoded:     {[encoding.decode([t]) for t in tokens]}")
print(f"Ratio:       {len(tokens) / len(text.split()):.2f} tokens per word")

In [ ]:
# ── 1b. Compare encodings across models ───────────────────────────────────
# Different models use different tokenizers — token counts vary!

samples = {
    "English prose": "The quick brown fox jumps over the lazy dog.",
    "Python code":   "def fibonacci(n):\n    return n if n <= 1 else fibonacci(n-1) + fibonacci(n-2)",
    "JSON payload":  '{"patient_id": "P-12345", "diagnosis": "Type 2 diabetes", "hba1c": 7.2}',
    "Medical term":  "Pneumonoultramicroscopicsilicovolcanoconiosis",
}

models = ["gpt-4o", "gpt-4", "gpt-3.5-turbo"]

print(f"{'Sample':<18} | " + " | ".join(f"{m:>16}" for m in models))
print("-" * 75)

for label, sample in samples.items():
    counts = []
    for m in models:
        enc = tiktoken.encoding_for_model(m)
        counts.append(len(enc.encode(sample)))
    row = " | ".join(f"{c:>16}" for c in counts)
    print(f"{label:<18} | {row}")

print("\nNotice: JSON uses far more tokens relative to its semantic content.")

In [ ]:
# ── 1c. Token-per-word ratio for different content types ──────────────────
for label, sample in samples.items():
    enc = tiktoken.encoding_for_model("gpt-4o")
    toks = enc.encode(sample)
    words = len(sample.split())
    ratio = len(toks) / words if words else float("inf")
    print(f"{label:<18}: {len(toks):>3} tokens / {words:>2} words = {ratio:.2f} tok/word")

---
## 2. Completion API Basics

The Completion API is the **single interface** through which all LLM-powered applications communicate with the model. Every agent framework, RAG pipeline, and chat application is ultimately a wrapper around this one API call.

Key fields to always log in production: `usage` (token counts), `finish_reason`, and model version.

In [ ]:
# ── 2a. Basic completion call and response inspection ─────────────────────
response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a concise medical terminology assistant."},
        {"role": "user", "content": "Define 'angioplasty' in one sentence."},
    ],
    temperature=0.2,
    max_tokens=100,
)

# The response object — study its structure
print("=== Response Content ===")
print(response.choices[0].message.content)
print()

print("=== Full Response Object ===")
print(f"Model:         {response.model}")
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Response ID:   {response.id}")
print(f"Created:       {response.created}")

In [ ]:
# ── 2b. Usage metadata and cost estimation ────────────────────────────────
usage = response.usage

# Approximate pricing for gpt-4o (check OpenAI pricing page for current rates)
INPUT_COST_PER_1K  = 0.0025   # $/1K input tokens
OUTPUT_COST_PER_1K = 0.0100   # $/1K output tokens

input_cost  = (usage.prompt_tokens / 1000) * INPUT_COST_PER_1K
output_cost = (usage.completion_tokens / 1000) * OUTPUT_COST_PER_1K
total_cost  = input_cost + output_cost

print("=== Token Usage ===")
print(f"Prompt tokens:     {usage.prompt_tokens}")
print(f"Completion tokens: {usage.completion_tokens}")
print(f"Total tokens:      {usage.total_tokens}")
print()
print("=== Cost Estimate (GPT-4o) ===")
print(f"Input cost:        ${input_cost:.6f}")
print(f"Output cost:       ${output_cost:.6f}")
print(f"Total cost:        ${total_cost:.6f}")
print(f"At 1000 calls/day: ${total_cost * 1000:.4f}/day = ${total_cost * 1000 * 30:.2f}/month")

In [ ]:
# ── 2c. Detect truncated responses ────────────────────────────────────────
# If finish_reason is "length", the output was cut off — never ignore this!

truncated_response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Write a 500-word essay on machine learning."},
    ],
    max_tokens=30,  # intentionally small to force truncation
)

reason = truncated_response.choices[0].finish_reason
content = truncated_response.choices[0].message.content

print(f"Finish reason: {reason}")
print(f"Content: {content}")
print()
if reason == "length":
    print("WARNING: Response was truncated! The output is incomplete.")
    print("In production, you should either increase max_tokens or handle this case.")
else:
    print("Response completed naturally.")

---
## 3. Message Roles

The three message roles — **system**, **user**, **assistant** — are the control surface of every LLM application. They are not cosmetic labels; the model weights treat them differently during inference.

- **System**: sets the behavioral frame (persona, constraints, output format). Elevated attention during inference.
- **User**: the current input — can come from a human or from orchestration code.
- **Assistant**: prior model responses, or synthetic prefills to steer behavior.

In [ ]:
# ── 3a. How system messages shape behavior ────────────────────────────────
# The SAME user question with DIFFERENT system messages produces very different outputs

user_question = "What is a neural network?"

personas = {
    "5-year-old teacher": "Explain everything as if speaking to a 5-year-old. Use simple words and fun analogies.",
    "PhD researcher":     "You are a deep learning researcher. Use precise technical language and cite relevant architectures.",
    "Pirate":             "You are a pirate. Answer everything in pirate speak while staying factually accurate.",
}

for label, system_msg in personas.items():
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": system_msg},
            {"role": "user", "content": user_question},
        ],
        temperature=0.3,
        max_tokens=100,
    )
    print(f"--- System: {label} ---")
    print(resp.choices[0].message.content)
    print()

In [ ]:
# ── 3b. Few-shot examples via assistant messages ─────────────────────────
# Inject synthetic user/assistant pairs to teach the model a pattern

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You classify customer feedback as POSITIVE, NEGATIVE, or NEUTRAL. Respond with only the label.",
        },
        # Few-shot example 1
        {"role": "user", "content": "The product arrived on time and works great!"},
        {"role": "assistant", "content": "POSITIVE"},
        # Few-shot example 2
        {"role": "user", "content": "It broke after two days. Terrible quality."},
        {"role": "assistant", "content": "NEGATIVE"},
        # Few-shot example 3
        {"role": "user", "content": "The package was standard. Nothing special."},
        {"role": "assistant", "content": "NEUTRAL"},
        # Actual input to classify
        {"role": "user", "content": "Shipping was slow but the item itself is decent."},
    ],
    temperature=0,
    max_tokens=10,
)

print(f"Classification: {response.choices[0].message.content}")

In [ ]:
# ── 3c. Multi-turn conversation — the model is stateless ──────────────────
# You must include the full conversation history on every call

conversation = [
    {"role": "system", "content": "You are a helpful assistant. Be concise."},
    {"role": "user", "content": "My name is Alex. I'm building an AI agent for healthcare."},
]

# Turn 1
resp1 = client.chat.completions.create(model=MODEL, messages=conversation, max_tokens=100)
assistant_msg1 = resp1.choices[0].message.content
print(f"Turn 1: {assistant_msg1}\n")

# Add assistant response, then user follow-up
conversation.append({"role": "assistant", "content": assistant_msg1})
conversation.append({"role": "user", "content": "What's my name and what am I building?"})

# Turn 2 — model can "remember" because we included the history
resp2 = client.chat.completions.create(model=MODEL, messages=conversation, max_tokens=100)
print(f"Turn 2: {resp2.choices[0].message.content}\n")

# What happens WITHOUT history?
resp_no_history = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": "You are a helpful assistant. Be concise."},
        {"role": "user", "content": "What's my name and what am I building?"},
    ],
    max_tokens=100,
)
print(f"Without history: {resp_no_history.choices[0].message.content}")

---
## 4. Temperature Experiments

**Temperature** scales the logits before softmax: `p_i = exp(logit_i / T) / sum(exp(logit_j / T))`.

- `T=0`: greedy decoding — deterministic, always picks the highest-probability token
- `T=1.0`: the model's native distribution
- `T>1.0`: flattened distribution — more creative, more random, potentially incoherent

For agent systems where reliability matters, use `temperature=0` (or near-zero). Save higher temperatures for creative tasks.

In [ ]:
# ── 4a. Same prompt at different temperatures ─────────────────────────────
prompt = "Suggest a creative name for an AI-powered coffee shop."
temperatures = [0, 0.5, 1.0, 1.5]

print("Prompt:", prompt)
print("=" * 60)

for temp in temperatures:
    # Run 3 times at each temperature to show variance
    outputs = []
    for _ in range(3):
        resp = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=temp,
            max_tokens=30,
        )
        outputs.append(resp.choices[0].message.content.strip())

    print(f"\nTemperature = {temp}")
    for i, out in enumerate(outputs, 1):
        print(f"  Run {i}: {out}")

    # Check if all outputs are identical (expected at temp=0)
    if len(set(outputs)) == 1:
        print("  => All identical (deterministic)")
    else:
        print(f"  => {len(set(outputs))} unique outputs (stochastic)")

In [ ]:
# ── 4b. Visualize temperature effect on probability distribution ──────────
# Conceptual demonstration: how temperature reshapes a probability distribution
import math

# Simulated logits for 8 candidate tokens
token_labels = ["the", "a", "an", "one", "this", "that", "some", "every"]
raw_logits   = [4.0, 2.5, 1.8, 1.2, 0.8, 0.5, 0.2, -0.5]

def softmax_with_temperature(logits, temperature):
    """Apply temperature scaling then softmax."""
    if temperature == 0:
        # Greedy: all probability on the max
        probs = [0.0] * len(logits)
        probs[logits.index(max(logits))] = 1.0
        return probs
    scaled = [l / temperature for l in logits]
    max_s = max(scaled)
    exps = [math.exp(s - max_s) for s in scaled]  # numerical stability
    total = sum(exps)
    return [e / total for e in exps]

print(f"{'Token':<8}", end="")
for t in [0, 0.5, 1.0, 1.5]:
    print(f"  T={t:<4}", end="")
print()
print("-" * 50)

for t in [0, 0.5, 1.0, 1.5]:
    probs = softmax_with_temperature(raw_logits, t)
    if t == 0:
        all_probs = {0: probs}
    else:
        all_probs[t] = probs

for i, label in enumerate(token_labels):
    print(f"{label:<8}", end="")
    for t in [0, 0.5, 1.0, 1.5]:
        p = all_probs[t][i]
        bar = "#" * int(p * 30)
        print(f"  {p:.3f} {bar:<10}", end="")
    print()

print("\nAt T=0, all probability mass is on 'the' (greedy decoding).")
print("As temperature rises, the distribution flattens — rare tokens get more chance.")

---
## 5. Streaming

Streaming lets you display tokens as they arrive instead of waiting for the full response. This is critical for user-facing applications where perceived latency matters. The API sends Server-Sent Events (SSE), and each chunk contains a delta with one or more tokens.

In [ ]:
# ── 5a. Streaming response — token by token ───────────────────────────────
import time

stream = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "user", "content": "Write a haiku about machine learning."},
    ],
    temperature=0.7,
    max_tokens=50,
    stream=True,
)

print("Streaming response:")
print("-" * 40)

collected_content = []
token_count = 0
start = time.time()

for chunk in stream:
    delta = chunk.choices[0].delta
    if delta.content:
        print(delta.content, end="", flush=True)
        collected_content.append(delta.content)
        token_count += 1

elapsed = time.time() - start
full_text = "".join(collected_content)

print(f"\n{'-' * 40}")
print(f"Chunks received: {token_count}")
print(f"Time to stream:  {elapsed:.2f}s")
print(f"Full text:       {full_text}")

---
## 6. Tool / Function Calling

Tool calling is the mechanism that bridges LLMs with external systems. The model does not execute the function — it generates a structured request indicating *which* tool to call and *what arguments* to pass. Your code executes the tool and feeds the result back.

This is the foundation of every agentic system: the **Reason → Act → Observe** loop.

In [ ]:
# ── 6a. Define a tool and make a tool-calling request ─────────────────────

# Step 1: Define the tool schema
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Get the current weather for a given city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {
                        "type": "string",
                        "description": "The city name, e.g. 'San Francisco'",
                    },
                    "units": {
                        "type": "string",
                        "enum": ["celsius", "fahrenheit"],
                        "description": "Temperature units",
                    },
                },
                "required": ["city"],
            },
        },
    }
]

# Step 2: Ask a question that should trigger the tool
messages = [
    {"role": "system", "content": "You are a helpful assistant. Use tools when needed."},
    {"role": "user", "content": "What's the weather like in Tokyo right now?"},
]

response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",
)

assistant_message = response.choices[0].message
print(f"Finish reason: {response.choices[0].finish_reason}")
print(f"Tool calls:    {assistant_message.tool_calls}")
print()

# Step 3: Parse the tool call
if assistant_message.tool_calls:
    tool_call = assistant_message.tool_calls[0]
    print(f"Function name: {tool_call.function.name}")
    print(f"Arguments:     {tool_call.function.arguments}")
    args = json.loads(tool_call.function.arguments)
    print(f"Parsed args:   {args}")

In [ ]:
# ── 6b. Execute the tool and feed the result back ─────────────────────────

# Simulate tool execution (in production this would call a real weather API)
def get_weather(city: str, units: str = "celsius") -> dict:
    """Simulated weather API."""
    mock_data = {
        "Tokyo":         {"temp": 22, "condition": "Partly cloudy", "humidity": 65},
        "San Francisco": {"temp": 16, "condition": "Foggy", "humidity": 78},
        "London":        {"temp": 12, "condition": "Rainy", "humidity": 85},
    }
    data = mock_data.get(city, {"temp": 20, "condition": "Clear", "humidity": 50})
    if units == "fahrenheit":
        data["temp"] = round(data["temp"] * 9 / 5 + 32)
    data["units"] = units
    data["city"] = city
    return data

# Execute the tool with the model's arguments
tool_result = get_weather(**args)
print(f"Tool result: {json.dumps(tool_result, indent=2)}\n")

# Feed the result back to the model
messages.append(assistant_message)  # include the assistant's tool-call message
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,
    "content": json.dumps(tool_result),
})

# Get the final response that incorporates the tool result
final_response = client.chat.completions.create(
    model=MODEL,
    messages=messages,
    tools=tools,
)

print("Final response (with tool result):")
print(final_response.choices[0].message.content)

---
## 7. Structured Outputs

Structured outputs guarantee that the model's response conforms to a specific JSON schema. This eliminates the need to parse free-text responses and handle malformed output — a major source of production failures.

We use `response_format` with a Pydantic model to define the expected schema.

In [ ]:
# ── 7a. Structured output with a Pydantic model ──────────────────────────
from pydantic import BaseModel
from typing import Literal

# Define the expected output schema
class MovieReview(BaseModel):
    title: str
    year: int
    genre: Literal["action", "comedy", "drama", "sci-fi", "horror", "other"]
    rating: float  # 0.0 to 10.0
    summary: str
    pros: list[str]
    cons: list[str]

# Request structured output
response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a movie critic. Analyze the given movie and return a structured review.",
        },
        {
            "role": "user",
            "content": "Review the movie 'Inception' by Christopher Nolan.",
        },
    ],
    response_format=MovieReview,
    temperature=0.3,
)

# The response is automatically parsed into our Pydantic model
review = response.choices[0].message.parsed

print(f"Title:   {review.title}")
print(f"Year:    {review.year}")
print(f"Genre:   {review.genre}")
print(f"Rating:  {review.rating}/10")
print(f"Summary: {review.summary}")
print(f"Pros:    {review.pros}")
print(f"Cons:    {review.cons}")
print()
print(f"Type:    {type(review).__name__}  (validated Pydantic model)")
print(f"JSON:    {review.model_dump_json(indent=2)}")

In [ ]:
# ── 7b. Structured output for data extraction ────────────────────────────
# A practical use case: extracting structured data from unstructured text

class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str
    company: str
    role: str

unstructured_text = """
Hi, I'm Sarah Chen and I work as a Senior ML Engineer at DataFlow Inc.
You can reach me at sarah.chen@dataflow.io or call me at (415) 555-0192.
Looking forward to discussing the partnership!
"""

response = client.beta.chat.completions.parse(
    model=MODEL,
    messages=[
        {"role": "system", "content": "Extract contact information from the text."},
        {"role": "user", "content": unstructured_text},
    ],
    response_format=ContactInfo,
    temperature=0,
)

contact = response.choices[0].message.parsed
print("Extracted contact info:")
for field, value in contact.model_dump().items():
    print(f"  {field:<10}: {value}")

---
## Exercises

Three exercises that reinforce the chapter's core concepts. Each builds on what you practiced above.

### Exercise 1: Token Budget Calculator (Conceptual)

Build a function `estimate_conversation_cost()` that takes a list of messages and returns:
- Total input tokens (using tiktoken locally — no API call needed)
- Estimated output tokens (assume a configurable ratio, e.g., 1.5x the user message tokens)
- Estimated cost at given per-token rates
- Whether the conversation fits within a given context window

This is the kind of utility every production system needs. Use it to answer: "If we have a 20-turn conversation averaging 150 words per turn, what does it cost with GPT-4o?"

In [ ]:
# ── Exercise 1: Token Budget Calculator ───────────────────────────────────
# TODO: Implement the function below

def estimate_conversation_cost(
    messages: list[dict],
    model: str = "gpt-4o",
    context_window: int = 128_000,
    output_ratio: float = 1.5,
    input_cost_per_1k: float = 0.0025,
    output_cost_per_1k: float = 0.0100,
) -> dict:
    """
    Estimate token usage and cost for a conversation without making an API call.

    Args:
        messages: List of {"role": ..., "content": ...} dicts
        model: Model name (for selecting the right tokenizer)
        context_window: Max tokens the model supports
        output_ratio: Estimated output tokens as multiple of user message tokens
        input_cost_per_1k: Cost per 1K input tokens
        output_cost_per_1k: Cost per 1K output tokens

    Returns:
        Dict with input_tokens, estimated_output_tokens, total_cost, fits_in_context
    """
    # YOUR CODE HERE
    pass


# Test it:
# test_messages = [
#     {"role": "system", "content": "You are a helpful assistant."},
#     {"role": "user", "content": "Explain quantum computing in simple terms."},
# ]
# result = estimate_conversation_cost(test_messages)
# print(result)

### Exercise 2: Temperature Sweep with Statistics (Coding)

Write code that:
1. Runs a prompt 10 times at each of 5 temperature values (0, 0.3, 0.7, 1.0, 1.5)
2. Collects all responses
3. Computes: number of unique responses, average response length (in tokens), and lexical diversity (unique words / total words)
4. Prints a summary table

Use a short prompt like "Name one fruit" so the calls are cheap and fast. This exercise builds intuition for how temperature affects reliability in production systems.

In [ ]:
# ── Exercise 2: Temperature Sweep with Statistics ─────────────────────────
# TODO: Implement the temperature sweep

# Hint:
# temperatures = [0, 0.3, 0.7, 1.0, 1.5]
# prompt = "Name one fruit."
# runs_per_temp = 10
#
# For each temperature:
#   1. Collect `runs_per_temp` responses
#   2. Count unique responses
#   3. Compute average token length using tiktoken
#   4. Compute lexical diversity: len(unique_words) / len(total_words)
#
# Print a table like:
#   Temp  | Unique | Avg Tokens | Lexical Diversity
#   0.0   |    1   |     3.0    | 1.00
#   0.3   |    2   |     3.2    | 0.85
#   ...

# YOUR CODE HERE
pass

### Exercise 3: Model Routing Architecture (Design)

Design a `ModelRouter` class that selects the best model for a given request based on:
- **Task complexity**: simple Q&A vs. multi-step reasoning vs. creative writing
- **Token budget**: route to a cheaper model if the conversation is long
- **Latency requirements**: some tasks need fast responses (use a smaller model), others can wait

Implement the routing logic (no API calls needed). The router should:
1. Accept a request with metadata (task_type, max_latency_ms, max_cost)
2. Return the recommended model and explain why
3. Pre-compute the estimated cost using tiktoken

This mirrors a real production pattern: not every request needs GPT-4o.

In [ ]:
# ── Exercise 3: Model Routing Architecture ────────────────────────────────
# TODO: Implement the ModelRouter class

from dataclasses import dataclass

@dataclass
class RoutingRequest:
    messages: list[dict]
    task_type: str  # "simple_qa", "reasoning", "creative", "extraction"
    max_latency_ms: int = 5000
    max_cost_usd: float = 0.05

@dataclass
class RoutingDecision:
    model: str
    reason: str
    estimated_input_tokens: int
    estimated_cost_usd: float

class ModelRouter:
    """Routes requests to the optimal model based on task requirements."""

    # Model catalog with approximate characteristics
    MODELS = {
        "gpt-4o": {
            "input_cost_per_1k": 0.0025,
            "output_cost_per_1k": 0.0100,
            "avg_latency_ms": 3000,
            "capability": "high",
        },
        "gpt-4o-mini": {
            "input_cost_per_1k": 0.00015,
            "output_cost_per_1k": 0.0006,
            "avg_latency_ms": 1000,
            "capability": "medium",
        },
    }

    def route(self, request: RoutingRequest) -> RoutingDecision:
        """Select the best model for the given request."""
        # YOUR CODE HERE
        # Consider: task_type, token count, latency, and cost constraints
        pass


# Test it:
# router = ModelRouter()
# decision = router.route(RoutingRequest(
#     messages=[
#         {"role": "system", "content": "You are a helpful assistant."},
#         {"role": "user", "content": "What is 2 + 2?"},
#     ],
#     task_type="simple_qa",
#     max_latency_ms=2000,
#     max_cost_usd=0.001,
# ))
# print(f"Model: {decision.model}")
# print(f"Reason: {decision.reason}")